### Rank Order, KS Statistic calculation


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
pd.set_option('display.float_format', lambda x: '%.2f' % x)

df = pd.read_csv('data.csv')
df

,Borrower Name,Default Probability,Default Truth
0,Priya Rao,0.32,0
1,Raj Patel,0.67,1
2,Meera Gupta,0.56,1
3,Linda Johnson,0.18,0
4,Aditi Sharma,0.75,1
5,John Smith,0.49,1
6,Arjun Reddy,0.12,0
7,Emily Chen,0.44,0
8,Laura Kim,0.06,0
9,Vivek Singh,0.39,0


In [2]:
# sort the data by Default Probability in descending order
df_sorted = df.sort_values(by='Default Probability', ascending=False)
df_sorted

,Borrower Name,Default Probability,Default Truth
4,Aditi Sharma,0.75,1
1,Raj Patel,0.67,1
12,Anil Kumar,0.62,0
2,Meera Gupta,0.56,1
13,Sarah Lee,0.52,1
5,John Smith,0.49,1
7,Emily Chen,0.44,0
9,Vivek Singh,0.39,0
0,Priya Rao,0.32,0
10,Michael Brown,0.28,0


In [3]:
df_sorted['Quartile'] = pd.qcut(df_sorted['Default Probability'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_sorted

,Borrower Name,Default Probability,Default Truth,Quartile
4,Aditi Sharma,0.75,1,Q4
1,Raj Patel,0.67,1,Q4
12,Anil Kumar,0.62,0,Q4
2,Meera Gupta,0.56,1,Q4
13,Sarah Lee,0.52,1,Q3
5,John Smith,0.49,1,Q3
7,Emily Chen,0.44,0,Q3
9,Vivek Singh,0.39,0,Q3
0,Priya Rao,0.32,0,Q2
10,Michael Brown,0.28,0,Q2


In [4]:
df_grouped = df_sorted.groupby('Quartile').apply(lambda x: pd.Series({
    'Minimum Probability': x['Default Probability'].min(),
    'Maximum Probability': x['Default Probability'].max(),
    'Events': x['Default Truth'].sum(),
    'Non-Events': x['Default Truth'].count() - x['Default Truth'].sum(),
}))
df_grouped.reset_index(inplace=True)
df_grouped

C:\Users\hapatil\AppData\Local\Temp\ipykernel_2272\4269028133.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_grouped = df_sorted.groupby('Quartile').apply(lambda x: pd.Series({
C:\Users\hapatil\AppData\Local\Temp\ipykernel_2272\4269028133.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_grouped = df_sorted.groupby('Quartile').apply(lambda x: pd.Series({


,Quartile,Minimum Probability,Maximum Probability,Events,Non-Events
0,Q1,0.03,0.15,0.00,4.00
1,Q2,0.18,0.32,0.00,4.00
2,Q3,0.39,0.52,2.00,2.00
3,Q4,0.56,0.75,3.00,1.00


In [5]:
df_grouped.sort_values(by='Quartile', ascending=False, inplace=True)
df_grouped

,Quartile,Minimum Probability,Maximum Probability,Events,Non-Events
3,Q4,0.56,0.75,3.00,1.00
2,Q3,0.39,0.52,2.00,2.00
1,Q2,0.18,0.32,0.00,4.00
0,Q1,0.03,0.15,0.00,4.00


In [6]:
df_grouped['Event Rate'] = df_grouped['Events']*100 / (df_grouped['Events'] + df_grouped['Non-Events'])
df_grouped['Non-Events'] = df_grouped['Non-Events']*100 / (df_grouped['Events'] + df_grouped['Non-Events'])
df_grouped

,Quartile,Minimum Probability,Maximum Probability,Events,Non-Events,Event Rate
3,Q4,0.56,0.75,3.00,25.00,75.00
2,Q3,0.39,0.52,2.00,50.00,50.00
1,Q2,0.18,0.32,0.00,100.00,0.00
0,Q1,0.03,0.15,0.00,100.00,0.00


In [7]:
df_grouped['Cum Events'] = df_grouped['Events'].cumsum()
df_grouped['Cum Non-Events'] = df_grouped['Non-Events'].cumsum()
df_grouped

,Quartile,Minimum Probability,Maximum Probability,Events,Non-Events,Event Rate,Cum Events,Cum Non-Events
3,Q4,0.56,0.75,3.00,25.00,75.00,3.00,25.00
2,Q3,0.39,0.52,2.00,50.00,50.00,5.00,75.00
1,Q2,0.18,0.32,0.00,100.00,0.00,5.00,175.00
0,Q1,0.03,0.15,0.00,100.00,0.00,5.00,275.00


In [8]:
df_grouped['Cum Event Rate'] = df_grouped['Cum Events']*100 / df_grouped['Events'].sum()
df_grouped['Cum Non-Event Rate'] = df_grouped['Cum Non-Events']*100 / df_grouped['Non-Events'].sum()
df_grouped

,Quartile,Minimum Probability,Maximum Probability,Events,Non-Events,Event Rate,Cum Events,Cum Non-Events,Cum Event Rate,Cum Non-Event Rate
3,Q4,0.56,0.75,3.00,25.00,75.00,3.00,25.00,60.00,9.09
2,Q3,0.39,0.52,2.00,50.00,50.00,5.00,75.00,100.00,27.27
1,Q2,0.18,0.32,0.00,100.00,0.00,5.00,175.00,100.00,63.64
0,Q1,0.03,0.15,0.00,100.00,0.00,5.00,275.00,100.00,100.00


In [9]:
df_grouped['KS Statistic'] = np.abs(df_grouped['Cum Event Rate'] - df_grouped['Cum Non-Event Rate'])
df_grouped

,Quartile,Minimum Probability,Maximum Probability,Events,Non-Events,Event Rate,Cum Events,Cum Non-Events,Cum Event Rate,Cum Non-Event Rate,KS Statistic
3,Q4,0.56,0.75,3.00,25.00,75.00,3.00,25.00,60.00,9.09,50.91
2,Q3,0.39,0.52,2.00,50.00,50.00,5.00,75.00,100.00,27.27,72.73
1,Q2,0.18,0.32,0.00,100.00,0.00,5.00,175.00,100.00,63.64,36.36
0,Q1,0.03,0.15,0.00,100.00,0.00,5.00,275.00,100.00,100.00,0.00
